# Bus Halt Passenger Queue Simulation

## 1. Project title and introduction

This project compares passenger waiting and service performance at a bus halt with one and two boarding bays. The one-bay case is represented by an M/M/1 queue and the two-bay case by an M/M/2 queue.

A discrete-event simulation will advance the model using passenger arrivals and boarding departures. NumPy provides random sampling, Pandas organises experiment records, Matplotlib creates graphs, and SciPy can support statistical calculations. The current notebook contains the academic structure and implementation framework; no simulation results are fabricated.

In [1]:
# Google Colab setup; uncomment when dependencies are not already available.
# !pip install numpy pandas matplotlib scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Reproducible seed:", SEED)

NumPy: 2.5.5
Pandas: 3.1.1
Reproducible seed: 42


## 2. Problem formulation

Passengers arrive independently at a bus halt and join a queue when all boarding bays are occupied. The model needs to determine how a change from one to two bays affects passenger waiting and queue-building behaviour.

The problem is formulated as a comparison of two stochastic queue configurations under controlled arrival and service rates:

## 3. Objectives

The objectives are to:

1. Represent passenger arrivals using a Poisson process.
2. Represent boarding service times using exponential distributions.
3. Develop reproducible M/M/1 and M/M/2 discrete-event simulations.
4. Measure waiting time, queue length, utilisation, throughput, and system time.
5. Compare the simulation configurations across controlled experiments.
6. Verify key implementation behaviour and provide uncertainty estimates.
7. Explain the limitations of the assumptions and interpret the final results.

## 4. Queueing theory background

An M/M/c queue has $c$ independent identical servers. Passengers arrive according to a Poisson process at rate $\lambda$, and each boarding bay completes service according to an exponential distribution with rate $\mu$.

For M/M/1, utilisation is

$$\rho_1 = \frac{\lambda}{\mu}.$$ 

For M/M/2, utilisation is

$$\rho_2 = \frac{\lambda}{2\mu},$$

where the two-bay system is stable when $\lambda < 2\mu$. Erlang C gives the probability that an arriving passenger must wait when all $c$ bays are busy.

## 5. Model assumptions

The model structure will state and justify the following assumptions:

- Passenger arrivals follow a stationary Poisson process.
- All arrival and service observations are independent.
- Boarding service times are exponential and independent of arrival times.
- Passenger movement follows a first-in, first-out queue discipline.
- Each boarding bay serves one passenger at a time.
- The model has no deliberate balking, reneging, or priority rules unless added and analysed separately.
- A warm-up period may be excluded when estimating steady-state performance.

These assumptions are simplified. Their effect on real bus-halt behaviour will be discussed in the conclusions.

## 6. Mathematical equations

Let inter-arrival time $A$, arrival rate $\lambda$, service time $S$, and service rate $\mu$. The exponential distributions are

$$A \sim \operatorname{Exp}(\lambda), \qquad f_A(a)=\lambda e^{-\lambda a},$$

$$S \sim \operatorname{Exp}(\mu), \qquad f_S(s)=\mu e^{-\mu s}.$$ 

For a passenger $i$, total time in the system is

$$W_i^{system}=W_i^{queue}+S_i,$$

where $W_i^{queue}$ is waiting before boarding. For M/M/1, steady-state mean queue length and mean waiting time are represented analytically later as

$$L_q=\frac{\rho^2}{1-\rho}, \qquad W_q=\frac{L_q}{\lambda}.$$

For M/M/2, Erlang C provides the steady-state probability that an arriving passenger waits. The exact formulas and implementation will be added only during the simulation stage.

## 7. Simulation parameters

Parameters will be stored in a small Python dictionary or structured experiment table. Suggested parameters are described below; numerical values should remain editable and should not be interpreted as study results.

In [2]:
# Parameter framework. Values are configuration placeholders for later implementation.
simulation_parameters = {
    "arrival_rate_per_hour": None,
    "service_rate_per_hour": None,
    "simulation_hours": None,
    "warm_up_hours": None,
    "replications": None,
    "random_seed": SEED,
}

pd.DataFrame([simulation_parameters])

,arrival_rate_per_hour,service_rate_per_hour,simulation_hours,warm_up_hours,replications,random_seed
0,None,None,None,None,None,42


## 8. Random number generation

All stochastic samples will use a NumPy `Generator` created with a fixed seed. Using a named seed makes reruns reproducible and allows separate scenarios to receive controlled independent streams.

This section is reserved for seed management and random-stream design. It will not record generated passenger data as model results.

In [3]:
def create_random_stream(seed: int) -> np.random.Generator:
    """Create an independent reproducible NumPy random stream."""
    return np.random.default_rng(seed)

def create_seed_list(number_of_replications: int, base_seed: int = SEED):
    """Create one seed for each planned replication."""
    return list(np.arange(base_seed, base_seed + number_of_replications, dtype=int))

create_seed_list(number_of_replications=5)

[np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46)]

## 9. Passenger arrival generation

Passenger arrivals can be generated from exponential inter-arrival times:

$$T_{k+1}=T_k+A_k, \qquad A_k \sim \operatorname{Exp}(\lambda).$$

For inverse transform sampling, draw $U\sim U(0,1)$ and generate each inter-arrival time as

$$A_k=-\frac{\ln(U_k)}{\lambda}, \qquad T_{k+1}=T_k+A_k.$$

The function below is reproducible with a NumPy integer seed. It stops before or at the specified simulation duration and does not generate timestamps beyond that horizon.

In [4]:
def generate_arrival_times(arrival_rate, duration, seed=42):
    """Generate arrival timestamps using inverse transform sampling."""
    if isinstance(arrival_rate, bool) or not isinstance(arrival_rate, (int, float)):
        raise TypeError("arrival_rate must be a real number")
    if isinstance(duration, bool) or not isinstance(duration, (int, float)):
        raise TypeError("duration must be a real number")

    arrival_rate = float(arrival_rate)
    duration = float(duration)

    if not np.isfinite(arrival_rate) or arrival_rate <= 0:
        raise ValueError("arrival_rate must be a finite positive number")
    if not np.isfinite(duration) or duration < 0:
        raise ValueError("duration must be a finite non-negative number")

    if isinstance(seed, bool) or (seed is not None and not isinstance(seed, (int, np.random.Generator))):
        raise TypeError("seed must be an integer, a NumPy Generator, or None")
    if isinstance(seed, int) and seed < 0:
        raise ValueError("seed must be non-negative")

    rng = seed if isinstance(seed, np.random.Generator) else np.random.default_rng(seed)

    if duration == 0:
        return np.empty(0, dtype=float)

    timestamps = []
    elapsed = 0.0

    while True:
        uniforms = np.maximum(rng.random(1024), np.finfo(float).eps)
        inter_arrivals = -np.log(uniforms) / arrival_rate
        cumulative = elapsed + np.cumsum(inter_arrivals)
        timestamps.extend(cumulative[cumulative <= duration].tolist())

        if cumulative[-1] > duration:
            return np.asarray(timestamps, dtype=float)

        elapsed = float(cumulative[-1])

### Example generated arrival times

The example below uses an arrival rate of $\lambda=2$ passengers per time unit for a 10-unit observation period. The same seed reproduces the timestamps.

In [5]:
example_times = generate_arrival_times(arrival_rate=2.0, duration=10.0, seed=2026)

example_arrivals = pd.DataFrame({
    "passenger": np.arange(1, len(example_times) + 1),
    "arrival_time": example_times,
})

## 10. Passenger service-time generation

A passenger's boarding time can be generated from an exponential distribution:

$$S_i \sim \operatorname{Exp}(\mu).$$

For inverse transform sampling, draw $U_i\sim U(0,1)$ and calculate

$$S_i=-\frac{\ln(U_i)}{\mu},$$

where $μ$ is positive and the mean service duration is $1/μ$. The arrival rate $λ$ describes how often passengers enter the queue, while the service rate $μ$ describes how quickly one boarding bay serves passengers. In an M/M/2 system, the total service capacity is two times $μ$ when both bays are identical.

This component returns a NumPy array containing one positive duration per passenger. It accepts an integer seed or a NumPy `Generator` for reproducible experiments.

In [6]:
def generate_service_times(service_rate, passenger_count, seed=42):
    """Generate positive exponential service durations."""
    if isinstance(service_rate, bool) or not isinstance(service_rate, (int, float)):
        raise TypeError("service_rate must be a real number")
    if isinstance(passenger_count, bool) or not isinstance(passenger_count, int):
        raise TypeError("passenger_count must be an integer")

    service_rate = float(service_rate)
    if not np.isfinite(service_rate) or service_rate <= 0:
        raise ValueError("service_rate must be a finite positive number")
    if passenger_count < 0:
        raise ValueError("passenger_count must be non-negative")

    if isinstance(seed, bool) or (seed is not None and not isinstance(seed, (int, np.random.Generator))):
        raise TypeError("seed must be an integer, a NumPy Generator, or None")
    if isinstance(seed, int) and seed < 0:
        raise ValueError("seed must be non-negative")

    rng = seed if isinstance(seed, np.random.Generator) else np.random.default_rng(seed)
    if passenger_count == 0:
        return np.empty(0, dtype=float)

    uniforms = np.maximum(rng.random(passenger_count), np.finfo(float).eps)
    return -np.log(uniforms) / service_rate

### Small service-time example

For example, $μ=0.5$ passengers per time unit means a mean service duration of $1/0.5=2$ time units. The array below gives the generated duration for six passengers using a fixed seed.

In [7]:
example_service_times = generate_service_times(
    service_rate=0.5,
    passenger_count=6,
    seed=2026,
)

## 11. M/M/1 simulation

The M/M/1 section will model one boarding bay as a first-in, first-out queue. The state variables will include the next arrival time, next departure time, queue length, number in service, and clock time.

The implementation should compare arrival events with departure events and compute passenger waiting, service, and system times.

In [8]:
def simulate_mm1(arrival_rate, service_rate, duration, seed):
    """Reserved modular interface for the later M/M/1 simulation."""
    raise NotImplementedError("M/M/1 implementation is intentionally not included yet.")

## 12. M/M/2 simulation

The M/M/2 section will model two independent, identical boarding bays. Passenger arrivals wait only when both bays are busy. Each completed service makes the relevant bay available for the next queued passenger.

The implementation should use the same parameter definitions and performance calculations as M/M/1 so that the configurations remain comparable.

In [9]:
def simulate_mm2(arrival_rate, service_rate, duration, seed):
    """Reserved modular interface for the later M/M/2 simulation."""
    raise NotImplementedError("M/M/2 implementation is intentionally not included yet.")

## 13. Performance measurement

Planned performance measures include:

- mean passenger waiting time;
- mean queue length;
- mean time in the system;
- boarding-bay utilisation;
- throughput;
- maximum observed queue length; and
- waiting probability.

Replications will be stored as rows in a Pandas DataFrame. Reported values should include means and uncertainty measures rather than a single run.

In [10]:
performance_columns = [
    "boarding_bays",
    "replication",
    "arrival_rate",
    "service_rate",
    "mean_wait",
    "mean_queue_length",
    "mean_system_time",
    "utilisation",
    "throughput",
]

pd.DataFrame(columns=performance_columns)

,boarding_bays,replication,arrival_rate,service_rate,mean_wait,mean_queue_length,mean_system_time,utilisation,throughput


## 14. Verification

Verification is the process of checking that the implemented computer model represents the intended model logic correctly. The planned checks include:

- all generated inter-arrival and service times are positive and finite;
- passenger accounting is conserved between arrivals and completed service;
- queue length cannot become negative;
- utilisation and throughput are within logical bounds;
- identical seeds produce identical repeated runs;
- different seeds produce plausible variation; and
- limiting cases or analytical M/M/1 equations match sampled results within stated tolerances.

These checks must produce explicit pass/fail evidence before verification is discussed as complete.

## 15. Experiments

The experiment stage will vary arrival rates, service rates, run durations, and replication counts. Parameters should be selected to cover stable and unstable regions only if the instability is intentional and clearly interpreted.

The one-bay and two-bay models should be run under equivalent arrival and service parameters. Replication results should be retained so that confidence intervals and sensitivity analysis can be calculated.

## 16. Graphs and analysis

After successful implementation, Matplotlib will be used to create graphs comparing the M/M/1 and M/M/2 configurations. Suitable examples include:

- mean waiting time against arrival rate;
- mean queue length against arrival rate;
- utilisation against arrival rate;
- throughput against arrival rate; and
- confidence intervals or replications across parameter values.

No graph is populated here because it would require invented or unvalidated experimental output.

## 17. Conclusions

The final conclusions will be written only after simulation runs and verification checks are complete. They should state:

- the most important observed difference between one and two bays;
- whether and when adding a second bay provides meaningful waiting-time improvement;
- agreement or disagreement with analytical queueing formulas;
- uncertainty and replication evidence;
- practical limitations caused by the Markovian assumptions; and
- possible future extensions, such as observed passenger-arrival data or time-varying arrival rates.

No conclusion or measured improvement is claimed in this notebook structure.